In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

X_train = pd.read_csv('../data/X_train.csv', index_col='ROW_ID')
y_train = pd.read_csv('../data/y_train.csv', index_col='ROW_ID')
total_data = pd.concat([X_train, y_train], axis=1)

In [25]:
def add_beta_and_corr_features(X_train, X_test, RET_features):
    # --- BETA_k: sensitivity to the TS-level market mean ---
    for k in range(2, 21):
        cols = RET_features[:k]
        feat_name = f'BETA_{k}'

        for X in [X_train, X_test]:
            market = X.groupby('TS')[cols].transform('mean')
            R = X[cols].to_numpy(dtype=float)
            M = market[cols].to_numpy(dtype=float)

            Rm = R - R.mean(axis=1, keepdims=True)
            Mm = M - M.mean(axis=1, keepdims=True)
            cov = (Rm * Mm).mean(axis=1)
            var_m = (Mm ** 2).mean(axis=1)
            X[feat_name] = np.where(var_m == 0, 0.0, cov / var_m)

    # --- AVG_CORR_UNIV_k: average pairwise correlation with rest of universe that day ---
    for k in range(2, 21):
        cols = RET_features[:k]
        feat_name = f'AVG_CORR_UNIV_{k}'

        for X in [X_train, X_test]:
            X[feat_name] = 0.0
            for ts, idx in X.groupby('TS').groups.items():
                sub = X.loc[idx, cols].to_numpy(dtype=float)
                n_rows = sub.shape[0]
                if n_rows <= 1:
                    continue
                corr = np.corrcoef(sub)
                corr = np.nan_to_num(corr, nan=0.0)
                np.fill_diagonal(corr, 0.0)
                avg = corr.sum(axis=1) / (n_rows - 1)
                X.loc[idx, feat_name] = avg

    # --- ALLOCATIONS_SORTINO_3: downside-risk-adjusted 3-day mean, averaged across the day's universe ---
    cols = RET_features[:3]
    for X in [X_train, X_test]:
        R = X[cols].to_numpy(dtype=float)
        mean = R.mean(axis=1)
        downside = np.minimum(0.0, R)
        downside_std = np.sqrt((downside ** 2).mean(axis=1))
        X['SORTINO_3'] = mean / (downside_std + 1e-9)
        X['ALLOCATIONS_SORTINO_3'] = X.groupby('TS')['SORTINO_3'].transform('mean')

    # --- ALLOCATIONS_TREND_STRENGTH_10: (max-min)/std over 10 days, averaged across the day's universe ---
    cols = RET_features[:10]
    for X in [X_train, X_test]:
        window = X[cols]
        rng = window.max(axis=1) - window.min(axis=1)
        std = window.std(axis=1)
        X['TREND_STRENGTH_10'] = rng / (std + 1e-9)
        X['ALLOCATIONS_TREND_STRENGTH_10'] = X.groupby('TS')['TREND_STRENGTH_10'].transform('mean')

    new_cols = [
        f'BETA_{i}' for i in range(1, 21)
    ]
    return X_train, X_test, new_cols

In [27]:
x_train, x_test, new_cols = add_beta_and_corr_features(X_train.copy(), X_train.copy(), [f'RET_{i}' for i in range(1, 21)])
total_data = pd.concat([x_train, y_train], axis=1)
new_features = [f'BETA_{i}' for i in range(2,21)]

target_sign = np.sign(total_data['target'])

results = []
for col in new_features:
    corr = total_data[col].corr(total_data['target'])
    sign_match = (np.sign(total_data[col]) == target_sign).mean() * 100
    results.append({'feature': col, 'pearson_corr': corr, 'sign_match_pct': sign_match})

results_df = pd.DataFrame(results).sort_values('pearson_corr', key=abs, ascending=False)
print(results_df)

    feature  pearson_corr  sign_match_pct
1    BETA_3      0.008101       49.932931
12  BETA_14      0.004725       50.278804
11  BETA_13      0.004530       50.258503
9   BETA_11      0.004421       50.215435
10  BETA_12      0.004187       50.243894
13  BETA_15      0.004092       50.219989
15  BETA_17      0.003707       50.264954
14  BETA_16      0.003468       50.188684
8   BETA_10      0.002676       50.207087
16  BETA_18      0.002645       50.238202
18  BETA_20      0.002406       50.246930
5    BETA_7      0.002242       50.124746
17  BETA_19      0.002155       50.239530
3    BETA_5      0.002048       50.095148
7    BETA_9      0.002032       50.183371
4    BETA_6      0.001500       50.052839
6    BETA_8      0.001236       50.159845
2    BETA_4      0.000765       50.101789
0    BETA_2     -0.000113       49.978087
